# 06 · Handoffs and Multi-Agent Systems

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

A **handoff** is one agent passing the conversation to another: *"Let me
transfer you to billing."* It's the SDK's signature multi-agent primitive, and
together with agents-as-tools (notebook 04) it covers most multi-agent designs.
This notebook builds a support desk with handoffs, shows how to control what the
next agent sees, and ends with a check you should always add: **did the agent
actually do what it says it did?**

**You will learn**
- How handoffs work: they're tools that switch the active agent
- `handoff_description`, `last_agent`, and continuing a conversation after a handoff
- Customising handoffs: `on_handoff` callbacks and structured `input_type`
- **Input filters**: controlling the history the next agent sees
- **Verifying actions in code**: an agent claiming a refund it never issued
- Handoffs vs the manager pattern (agents as tools)

**Prerequisites:** notebooks 02 and 04.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

from pydantic import BaseModel
from agents import Agent, Runner, function_tool, handoff, RunContextWrapper
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from common import get_model

model = get_model()


def trace_run(result):
    for item in result.new_items:
        kind = type(item).__name__
        if kind == 'HandoffCallItem':
            print(f'  ⇢ handoff: {item.raw_item.name}')
        elif kind == 'ToolCallItem':
            print(f'  → {item.agent.name} calls {item.raw_item.name}({item.raw_item.arguments})')
        elif kind == 'ToolCallOutputItem':
            print(f'  ← {item.output}')
    print(f'  [{result.last_agent.name}] {result.final_output[:150]}')

## 1 · Why handoffs

Notebook 05 routed tickets with code: classify, then pick a specialist. That's
the right choice when routing is simple and one-shot. Handoffs are for when the
**conversation itself** should move: the specialist then talks to the user
directly for the rest of the conversation, with its own instructions and tools,
and can hand off again if needed.

```
 user ─► Triage ──transfer_to_billing──► Billing ──(its tools)──► reply to the user
                 └─transfer_to_tech─────► Tech    ──(its tools)──► reply to the user
```

**How it works:** each handoff is presented to the model as a **tool**
(`transfer_to_billing`). When the model calls it, the Runner switches the active
agent and continues the loop with the new agent's instructions and tools. The
conversation so far goes along. There's no separate routing step; the model's
choice *is* the routing.

## 2 · A support desk with handoffs

In [ ]:
refunds_issued = []                      # what REALLY happened (our "database")

@function_tool
def issue_refund(order_id: str) -> str:
    """Refund an order."""
    refunds_issued.append(order_id)
    return f'Refund issued for order {order_id}.'

@function_tool
def reset_password(email: str) -> str:
    """Send a password-reset link."""
    return f'Reset link sent to {email}.'


billing = Agent(name='Billing', model=model, tools=[issue_refund],
                handoff_description='Refunds, charges and invoices.',
                instructions='You are billing support. Resolve the issue with your tools, then reply to the customer.')
tech = Agent(name='Tech', model=model, tools=[reset_password],
             handoff_description='Login problems and technical issues.',
             instructions='You are tech support. Resolve the issue with your tools, then reply to the customer.')

triage = Agent(name='Triage', model=model, handoffs=[billing, tech],
               instructions=f'{RECOMMENDED_PROMPT_PREFIX}\n'
                            'You are the triage desk. Never solve problems yourself: hand off to the right specialist.')

for question in ['I cannot log in, my email is sam@example.com.', 'Please refund order A-17, it arrived broken.']:
    print(question)
    trace_run(await Runner.run(triage, question))
    print()

**Read the replies carefully before moving on.** The routing worked (each
question reached the right specialist), but did the specialists *do* anything?
In our runs they often didn't. Tech replied *"I need to transfer you to a tech
support specialist"*, although it **is** tech support, and Billing said the
request "has been transferred to the billing team" without issuing a refund.
They had just received a history full of transfer messages and carried on
talking about transfers instead of using their tools. We'll fix that in §5 and
learn to detect it in §6.

Other things to notice:

- **`handoff_description`** is what the *triage* model reads when deciding where
  to send the conversation, so write it for that reader.
- **`RECOMMENDED_PROMPT_PREFIX`** is a short system-prompt preamble shipped with
  the SDK. It explains handoffs to the model, and helps it use them reliably.
- **`result.last_agent`** tells you who ended up owning the conversation.

## 3 · Continuing the conversation after a handoff

The customer replies. Who should answer? The agent that **now owns** the
conversation, not the triage desk again. Start the next run from
`result.last_agent`, with the history from `to_input_list()`:

In [ ]:
first = await Runner.run(triage, 'I cannot log in, my email is sam@example.com.')
print('after turn 1, owner:', first.last_agent.name)

follow_up = first.to_input_list() + [{'role': 'user', 'content': 'Got it. Can you also send it to my backup: sam@home.net?'}]
second = await Runner.run(first.last_agent, follow_up)          # ← continue with the current owner
trace_run(second)

Restarting from `triage` would work too, but it costs an extra hop every turn,
and triage might route differently mid-conversation. (Sessions, notebook 09,
store the history, but *you* still choose which agent starts each run.)

## 4 · Customising a handoff

Wrapping an agent in `handoff(...)` lets you change how the transfer works:

| Option | What it does |
|---|---|
| `tool_name_override`, `tool_description_override` | rename or re-describe the transfer tool |
| `on_handoff=fn` | run your code at the moment of transfer (logging, pre-loading data, alerting a human) |
| `input_type=Model` | make the model pass **structured data** with the handoff (e.g. a reason) |
| `input_filter=fn` | change the history the next agent receives (§5) |
| `is_enabled=` | only offer the handoff in some situations |

An escalation handoff that must carry a reason and a priority, logged by our code:

In [ ]:
class Escalation(BaseModel):
    reason: str
    priority: int                 # 1 (low) – 3 (urgent)

escalations = []

async def log_escalation(ctx: RunContextWrapper, data: Escalation):
    escalations.append(data)      # e.g. create a ticket in your helpdesk system
    print(f'  [on_handoff] escalation logged: priority {data.priority}: {data.reason}')

manager = Agent(name='Duty manager', model=model,
                instructions='You are the duty manager. Apologise, and promise a callback within the hour.')

front_desk = Agent(name='Front desk', model=model,
                   instructions=f'{RECOMMENDED_PROMPT_PREFIX}\nHelp customers. If a customer is angry or '
                                'threatens to cancel, escalate to the duty manager with a reason and priority 1-3.',
                   handoffs=[handoff(manager, on_handoff=log_escalation, input_type=Escalation)])

trace_run(await Runner.run(front_desk, 'This is the third broken robot this month. I am cancelling our contract!'))

## 5 · What the next agent sees: input filters

By default the receiving agent gets **the whole conversation**, including the
previous agent's internal tool calls and the handoff call itself. That's often
fine, but it can confuse the new agent. It sees tool calls to tools it doesn't
have, and conversation that isn't relevant to its job. In the LangGraph version
of this desk, a receiving agent even imitated the previous agent's transfer call.

An **input filter** rewrites the history at the moment of transfer. The SDK
ships some, e.g. `handoff_filters.remove_all_tools` (drop every tool call and
result, keep the conversation), and you can write your own function:

```python
handoff(billing, input_filter=handoff_filters.remove_all_tools)
```

The same desk as §2, with both handoffs filtered:

In [ ]:
filtered_triage = triage.clone(handoffs=[
    handoff(billing, input_filter=handoff_filters.remove_all_tools),
    handoff(tech, input_filter=handoff_filters.remove_all_tools),
])

for question in ['I cannot log in, my email is sam@example.com.', 'Please refund order A-17, it arrived broken.']:
    print(question)
    trace_run(await Runner.run(filtered_triage, question))
    print()

Now each specialist sees the customer's words, not the transfer machinery, and
gets on with its job.

## 6 · Did it actually do it? Verify actions in code

While testing this desk, we saw something important: the Billing agent replied
*"The refund for order A-17 has been processed successfully"* **without ever
calling `issue_refund`**. No refund existed; the customer was told there was one.

Across our test runs, the default (unfiltered) desk often skipped the tool,
and at least once claimed a refund that never happened. The filtered desk
called `issue_refund` every time (7 out of 7 runs).

That's still too few runs to *prove* the filter fixes the problem, and you
shouldn't rely on any prompt or filter alone. The lesson is more basic: **an agent's words are not
evidence that an action happened.** The run records every tool call, so check
it in code before you tell the customer anything:

In [ ]:
def called(result, tool_name: str) -> bool:
    # True if the run really executed this tool (not just talked about it).
    return any(type(i).__name__ == 'ToolCallItem' and i.raw_item.name == tool_name for i in result.new_items)


for attempt in range(1, 4):
    refunds_issued.clear()
    result = await Runner.run(filtered_triage, 'Please refund order A-17, it arrived broken.')
    claims = 'refund' in result.final_output.lower()
    really = called(result, 'issue_refund')
    verdict = 'OK' if really else ('⚠️  CLAIMED BUT NOT DONE: do not send this reply' if claims else 'no refund')
    print(f'run {attempt}: tool called={really}, database={refunds_issued} → {verdict}')

In production, a claim without a matching action should never reach the
customer. Retry, route to a human, or reply with what actually happened. This
check costs nothing and catches a failure no prompt can rule out.

## 7 · Handoffs vs the manager pattern

The other way to combine agents is the **manager pattern**: one agent keeps
control and calls specialists as tools (`agent.as_tool`, notebook 04).

| | Handoffs | Manager (agents as tools) |
|---|---|---|
| Who talks to the user at the end | the specialist | the manager |
| Control after the specialist finishes | stays with the specialist | returns to the manager |
| Combining several specialists' results | awkward | natural |
| Specialist sees | the conversation (filterable) | only the input the manager writes |
| Good for | triage, "transfer me to…" flows | tasks needing several experts, one answer |

Use **handoffs** when the conversation should *belong* to a specialist. Use the
**manager** when one agent must gather and combine several specialists' work.
Mixing is normal: a triage desk hands off to a specialist that uses other agents
as tools. And for anything whose order is fixed, orchestrate in code (notebook 05).

## 8 · Common mistakes

- **Vague `handoff_description`s**, so triage routes badly. Describe what each specialist handles.
- **Restarting every turn at triage.** Continue from `result.last_agent`.
- **Trusting what an agent says it did.** Verify side effects from the run's items (or your database).
- **Letting agents see irrelevant tool traffic.** Use an input filter.
- **Handoff loops** (A → B → A …). Give agents clear, non-overlapping scopes; `max_turns` is the backstop.
- **Using handoffs to combine results.** That's the manager pattern's job.

## 9 · Check yourself

<details><summary><b>1.</b> What is a handoff, mechanically?</summary>

A tool (e.g. `transfer_to_billing`) presented to the model. When the model calls it, the Runner switches the active agent to the target and continues the loop with that agent's instructions and tools, carrying the conversation along.
</details>

<details><summary><b>2.</b> After a handoff to Tech, the user sends a follow-up. Which agent should run it, and how?</summary>

The current owner: `Runner.run(result.last_agent, result.to_input_list() + [new_message])`.
</details>

<details><summary><b>3.</b> When would you give a handoff an <code>input_type</code>?</summary>

When the transfer should carry structured data your code needs, such as an escalation reason and priority. The model must fill it, and `on_handoff` receives it as a validated object.
</details>

<details><summary><b>4.</b> An agent replies "your refund has been processed". How do you know whether it's true?</summary>

Check the run's items for a `ToolCallItem` of the refund tool (and its successful output), or check your system of record. The agent's words aren't evidence; only the recorded tool call is.
</details>

<details><summary><b>5.</b> Handoffs or manager pattern for "answer a question that needs a price lookup and a calculation"?</summary>

The manager pattern (or code orchestration). The answer combines two specialists' results, and one agent must collect them. Handoffs suit "move this conversation to the right owner".
</details>

## Takeaway

A handoff is a tool that moves the conversation to another agent: describe
targets well, continue from `last_agent`, customise with `on_handoff`,
`input_type` and `input_filter`. Choose handoffs when the conversation should
belong to a specialist, and the manager pattern when results must be combined.
And never trust an agent's claim about a side effect: **verify it from the run.**

Next → `07_guardrails.ipynb`: stopping bad input and output before they do damage.